In [1]:
__author__ = 'Mary T. Yohannes'
# personal script modified by Nico

# This script fits the null logistic using a full GRM using SAIGE
## https://saigegit.github.io/SAIGE-doc/docs/single_step1.html
# running on assist_khat_amt NA values = 0 based model
# GP-filtered based model


In [9]:
import hail as hl
import hailtop.batch as hb

from hailtop import fs


In [10]:
# get file size to allocate resources for batch jobs accordingly
def get_file_size(file):
    file_info = hl.utils.hadoop_stat(file)
    size_bytes = file_info['size_bytes']
    size_gigs = size_bytes / (1024 * 1024 * 1024)
    return size_gigs

In [11]:
# fit the null logistic/linear mixed model using a full GRM
def fit_null_model(b, site, plink_files, phenotype_file, phenotype_col, covariate_cols, trait_type, invNormalize, storage_size):
    j = b.new_job(name=f'{site} - {phenotype_col}')
    j.image('wzhou88/saige:1.3.0') # SAIGE docker image
    j.cpu(16)
    j.storage(f'{storage_size}Gi')  # increase storage according to file size

    # model and variance ratio files - inputs for step 4 single-variant association tests (SAIGE step 2 - used in step4_association_test_khat_saige2.py script)
    j.declare_resource_group(ofile={
        'rda': '{root}.rda',
        'varianceRatio.txt': '{root}.varianceRatio.txt'})

    j.command(f'''
    step1_fitNULLGLMM.R \
    --plinkFile={plink_files} \
    --phenoFile={phenotype_file} \
    --phenoCol={phenotype_col} \
    --covarColList={covariate_cols} \
    --sampleIDColinphenoFile=IID \
    --traitType={trait_type} \
    --outputPrefix={j.ofile} \
    --nThreads=16 \
    --LOCO=True \
    --invNormalize={invNormalize} ''')

    return j


In [16]:
if __name__ == '__main__':

    backend = hb.ServiceBackend(
        billing_project='neale-pumas-bge',
        remote_tmpdir='gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp'
    )

    phenotype_cols = ["assist_khat"]

    for phenotype in phenotype_cols:

        b = hb.Batch(backend=backend, name=f'Step3: null_saige - GP-filtered - {phenotype}')

        sites = ["KEMRI", "Moi", "UCT"]

        for site in sites:

            plink_files = b.read_input_group(
                bed=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.bed',
                bim=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.bim',
                fam=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.fam'
            )

            phenotype_file = b.read_input(
                f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/phenotype_files/special_char_change/no_hyphen/w_site/StudySite_w_MatchingIIDs/no_char_{site}_khat_pheno_subset_lerato_with_studysite.tsv'
            )

            plink_size = get_file_size(
                'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_AAU_ldpruned.bed'
            )
            storage_size = round(plink_size + 3)

            covariate_cols = "PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,age,sex"

            if phenotype == "assist_khat":
                trait_type = "binary"
                invNormalize = "FALSE"

            run_fit_null = fit_null_model(
                b, site, plink_files, phenotype_file, phenotype,
                covariate_cols, trait_type, invNormalize, storage_size
            )

            b.write_output(
                run_fit_null.ofile,
                f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/saige_step1_outputs/gwaspy/{phenotype}/gp08_{site}_{phenotype}_saige_null'
            )

        b.run(wait=False)

    backend.close()


/var/folders/11/84m9v1z9659br_10hygjq6n80000gp/T/ipykernel_58926/3810196495.py:3: DeprecationWarning:

Call to deprecated function (or staticmethod) hadoop_stat. (Prefer hailtop.fs.stat) -- Deprecated since version 0.2.137.



Output()

In [17]:
if __name__ == '__main__':

    backend = hb.ServiceBackend(
        billing_project='neale-pumas-bge',
        remote_tmpdir='gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp'
    )

    phenotype_cols = ["assist_khat"]

    for phenotype in phenotype_cols:

        b = hb.Batch(backend=backend, name=f'Step3: null_saige - GP-filtered w/ study_site - {phenotype}')

        sites = ["Uganda", "AAU"]

        for site in sites:

            plink_files = b.read_input_group(
                bed=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.bed',
                bim=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.bim',
                fam=f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_{site}_ldpruned.fam'
            )

            phenotype_file = b.read_input(
                f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/phenotype_files/special_char_change/no_hyphen/w_site/StudySite_w_MatchingIIDs/no_char_{site}_khat_pheno_subset_lerato_with_studysite.tsv'
            )

            plink_size = get_file_size(
                'gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/gwaspy/gp08_AAU_ldpruned.bed'
            )
            storage_size = round(plink_size + 3)

            covariate_cols = "PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,age,sex,study_site"

            if phenotype == "assist_khat":
                trait_type = "binary"
                invNormalize = "FALSE"

            run_fit_null = fit_null_model(
                b, site, plink_files, phenotype_file, phenotype,
                covariate_cols, trait_type, invNormalize, storage_size
            )

            b.write_output(
                run_fit_null.ofile,
                f'gs://neurogap-bge-imputed-regional/nico/khat_gwas/saige_step1_outputs/gwaspy/{phenotype}/gp08_{site}_{phenotype}_saige_null'
            )

        b.run(wait=False)

    backend.close()


/var/folders/11/84m9v1z9659br_10hygjq6n80000gp/T/ipykernel_58926/3810196495.py:3: DeprecationWarning:

Call to deprecated function (or staticmethod) hadoop_stat. (Prefer hailtop.fs.stat) -- Deprecated since version 0.2.137.



Output()